# Part 2 - Advanced Modelling

Here we introduce some techniques for creating more advanced models


## Part 2.1 - Additional model creation options

Here we will briefly give examples of some more advanced model creation options beyond the basic workflow.

It is possible to make stoichiometries parameters. I.e. below we create a polymerisation model and define `n` as a parameter at the time of simulation:

In [ ]:
using Catalyst
polymerisation_model = @reaction_network begin
    (kB,kD), n*X <--> Xn
end

using OrdinaryDiffEq, Plots
u0 = [:X => 1.0, :Xn => 1.0]
ps = [:kB => 0.01, :kD => 0.1, :n => 4]
oprob = ODEProblem(polymerisation_model, u0, 20.0, ps)
osol = solve(oprob)
plot(osol)

Next, it is possible to use the `@parameters` and `@species` "options" to manually declare model parameters and species. This can be used to e.g. provide default values which are used at the time of simulation:

In [ ]:
bd_model = @reaction_network begin
    @species X(t) = 1.0
    @parameters p = 2.0  # d does not need to be listed here.
    (p,d), 0 <--> X
end

ps = [:d => 0.5]
oprob = ODEProblem(bd_model, [], 10.0, ps)
osol = solve(oprob)
plot(osol)

Do you remember this model used in the previous section?

In [ ]:
rn_14 = @reaction_network begin
    (p,d), 0 <--> E
    k*E, X_i --> X_a
end

What do we do if we do not want to define reactions for `E`, i.e. the enzyme have a constant level?

In [ ]:
rn_14 = @reaction_network begin
    k*E, X_i --> X_a
end

Here, `@reaction_network` will infer `E` to be a parameter (as it does not occur in any reaction). We can confirm this using

In [ ]:
parameters(rn_14)

If we want to specifically designate `E` to be a species, we can do this using the `@species` option.

In [ ]:
rn_14 = @reaction_network begin
    @species E(t)
    k*E, X_i --> X_a
end
parameters(rn_14)

If we simulate the model now, we define `E` in the initial condition vector, and we will also see its value in the plot.

In [ ]:
u0 = [:E => 2.0, :X_i => 10.0, :X_a => 0.0]
ps = [:k => 0.5]
oprob = ODEProblem(rn_14, u0, 10.0, ps)
osol = solve(oprob)
plot(osol)

## Part 2.2 - Non-reaction model components

The Catalyst `@reaction_network` DSL offers a range of options for adding non-reaction components to models, here, we will have a look at a few of them.

#### Observables

Observables are quantities that can be computed from the system state, but don't directly affect its dynamics. Primarily they act as easy quantities that can be plotted and queried. Below we create a dimerisation model, but make the total amount of `X` in the system an observable. Observables are decalred using the `@observables` option.

In [ ]:
dimerisation_model = @reaction_network begin
    @observables X_tot ~ X + 2X2
    (kB,kD), 2X <--> X2
end

We can now simulate the model using normal means. Specifically, we plot the amount of `X` and the total amount of `X` (the observable). As expected, the second quantity remains unchanged.

In [ ]:
u0 = [:X => 100.0, :X2 => 0.0]
ps = [:kB => 0.01, :kD => 0.1]
oprob = ODEProblem(dimerisation_model, u0, 10.0, ps)
osol = solve(oprob)
plot(osol; idxs = [:X, :X_tot])

#### Events

It is possible to add events to models. Here, under some condition, the system is updated (typically by changing some variable or parameter values). An event consists of a *condition* part (which determines when the event is triggered) and an *affect* part (which determines how the system changes when the event is triggered) We will consider four different types of events. Each have affect parts working in the same way, but their conditions work differently:
1. Continuous events. These are events that trigger whenever some equality relation holds.
2. Time-triggered discrete events. These are events that triggers at some predefined time points.
3. Periodic discrete events. These are events that triggers ar some regular time period.
4. Conditional discrete events. These are events that are checked at the end of each simulation timestep, and triggered if the condition is met.

We will consider each type in turn. There are also *callbacks* which permits defining more complicated conditions and affect, but these will not be considered in this workshop.

First we declare a continuous event. Here,we make one which add `1.0` unit of the species `X` whenever the concentration of `X` goes below a certain threshold. Continuous events are declared using the `@continuous_events` option.

In [ ]:
degradation_model = @reaction_network begin
    @continuous_events [X ~ 2.0] => [X => X + 1] # The event triggers when `[X ~ 2.0]` holds. When it does, `X + 1` is evaluated and put in `X`.
    d, X --> 0
end

u0 = [:X => 3.0]
ps = [:d => 2.0]
oprob = ODEProblem(degradation_model, u0, 2.0, ps)
osol = solve(oprob)
plot(osol)

Next we consider a time-triggered discrete event. here the condition is simply a vector of values, and the event is triggered at each of those time points.

In [ ]:
degradation_model = @reaction_network begin
    @discrete_events [2.0, 6.0, 7.0] => [X => X + 1]
    d, X --> 0
end

u0 = [:X => 3.0]
ps = [:d => 2.0]
oprob = ODEProblem(degradation_model, u0, 10.0, ps)
osol = solve(oprob)
plot(osol)

For periodic discrete events the condition is simply a number, with the event triggered repeatedly with that number as the period:

In [ ]:
degradation_model = @reaction_network begin
    @discrete_events 2.0 => [X => X + 1]
    d, X --> 0
end

u0 = [:X => 3.0]
ps = [:d => 2.0]
oprob = ODEProblem(degradation_model, u0, 10.0, ps)
osol = solve(oprob)
plot(osol)

Finally, conditional discrete events have a condition that is checked at the end of each time step, and if the condition hold, the event is triggered.

In [ ]:
degradation_model = @reaction_network begin
    @discrete_events (X < 1.0) => [X => X + 1]
    d, X --> 0
end

u0 = [:X => 3.0]
ps = [:d => 2.0]
oprob = ODEProblem(degradation_model, u0, 10.0, ps)
osol = solve(oprob)
plot(osol)

*In theory* this should be equivalent to the continuous event we used previously. However, for continuous events, the simulation integrator will find the exact timepoint when then threshold holds, and trigger the event. Here, it will simply trigger at the first timestep when the condition hold. We can see if we `X == 1.0` as the condition. *In theory* this will hold, however, the probability that it will happen at any given timestep is negligible. hence we can see that the event does not trigger at all.

In [ ]:
degradation_model = @reaction_network begin
    @discrete_events (X == 1.0) => [X => X + 1]
    d, X --> 0
end

u0 = [:X => 3.0]
ps = [:d => 2.0]
oprob = ODEProblem(degradation_model, u0, 10.0, ps)
osol = solve(oprob)
plot(osol)

Event affects can update parameters as well, use non-trivial update expressions, and update multiple quantities. Below we have an event which also updates the degradation parameter when it triggers.

In [ ]:
degradation_model = @reaction_network begin
    @parameters d_up
    @continuous_events [X ~ 1.0] => [X => X + 1, d => d / log(d_up + d)] # The update for d here does not make natural sense, but just show the kind of expressions one can build.
    d, X --> 0
end

u0 = [:X => 3.0]
ps = [:d => 10.0, :d_up => 1.0]
oprob = ODEProblem(degradation_model, u0, 2.0, ps)
osol = solve(oprob)
plot(osol)

If we want, we can plot the value of `d` throughout the simulation.

In [ ]:
plot(osol; idxs = :d)

We can also access its value using normal indexing syntax (generally, when indexing a parameter, we have to do `.ps` first).

In [ ]:
osol.ps[:d]

#### Coupled Equations
Catalyst enables the coupling of reaction network models with differential and algebraic equation models. These can be declared using an intuitive syntax and through the `@equations` option.

Here we consider a model of a cell growing in some nutrient medium. There exist a fixed quantity of nutrient (`N`) which is consumed. A protein that exist in an active and an inactive form (`Xᵢ` and `Xₐ`) determines the degradation rate of the nutrient, with presence of the nutrient driving the protein's activation. We declare the following model:

In [ ]:
nutrient_model = @reaction_network begin
    @parameters d
    @equations D(N) ~ - d*Xₐ*N
    (kₐ*N,kᵢ), Xᵢ <--> Xₐ
end

Here, every symbol appearing within the `@equation` option is assumed to be a variable. For the degradation rate (`d`) to be considered a parameter, we have to explicitly declare it as such. `D(N)` is automatically assumed to be the differential with respect to time (*dN/dt*). We can check which ODE this generates through

In [ ]:
ode_model(nutrient_model)

Next, we can simulate and plot it using normal syntax.

In [ ]:
u0 = [:N => 1.0, :Xᵢ => 1.0, :Xₐ => 0.0]
ps = [:d => 0.1, :kₐ => 0.5, :kᵢ => 0.1]
oprob = ODEProblem(nutrient_model, u0, 150.0, ps)
osol = solve(oprob)
plot(osol)

It is possible to have multiple equations in a model. Here, we can use a `begin ... end` statement and declare each equation on a single line. Here, we add a cell volume as well, which increased in the presence of `Xa`.

In [ ]:
nutrient_volume_model = @reaction_network begin
    @parameters d
    @equations begin
        D(N) ~ - d*Xₐ*N
        D(V) ~ 0.03*Xₐ
    end
    (kₐ*N,kᵢ), Xᵢ <--> Xₐ
end

u0 = [:N => 1.0, :V => 0.5, :Xᵢ => 1.0, :Xₐ => 0.0]
ps = [:d => 0.1, :kₐ => 0.5, :kᵢ => 0.1]
oprob = ODEProblem(nutrient_volume_model, u0, 150.0, ps)
osol = solve(oprob)
plot(osol)

Algebraic equations can also be coupled to ODE. Here, if we consider the following reaction network model

In [ ]:
dimerisation_model = @reaction_network begin
    (mmr(X2,v,K),d), 0 <--> X
    (kB,kD), 2X <--> X2
end

If the dynamics of the binding/unbinding reactions is much faster than the production/degradation reactions, then the `2X <--> X2` reaction can be assumed to be at steady state at the time scale of the production/degradation reaction. We can then rephrase our model using the (algebraic) steady state equation to define the concentration of $X2$:

In [ ]:
algebraic_dimerisation_model = @reaction_network begin
    @parameters kB kD
    @equations kB*X^2 ~ kD*X2
    (mmr(X2,v,K),d), 0 <--> X
end

When we simulate this model, `X2`'s initial condition can be computed using the equation from that of `X`. This, however, is carried out using a nonlinear solve call, for which we need an initial "guess" which we provide (generally, any guess value should work).

In [ ]:
u0 = [:X => 1.0]
guesses = [:X2 => 1.0]
ps = [:v => 2.0, :K => 1.0, :d => 0.2, :kB => 0.1, :kD => 0.4]
oprob = ODEProblem(algebraic_dimerisation_model, u0, 10.0, ps; guesses, mtkcompile = true) # `mtkcompile = true` is required for any models with algebraic equations.
osol = solve(oprob)
plot(osol)

## Part 2.3 - Programmatic modelling

So far we have used Catalyst's `@reaction_network` DSL to create models. However, there is an alternative approach. While a little bit more convoluted, *programmatic modelling* can also be very powerful. Furthermore, this also exposes some of the internals of Catalyst to the user. Here we will give a brief introduction to programmatic modelling in Catalyst.

#### Model components

Consider a normal Catalyst model. It consists of *species*, *parameters*, and *reactions*. Here we create a simple linear pathway model using the normal approach and check its content.

In [ ]:
linear_pathway = @reaction_network begin
    k, X0 --> X1
    k, X1 --> X2
    k, X2 --> X3
end

In [ ]:
species(linear_pathway)

In [ ]:
parameters(linear_pathway)

In [ ]:
reactions(linear_pathway)

As a brief parenthesis we mote that for am odel with mixed reaction network/equation structure we also have the `nonspecies` (for non-species variables) and `nonreaction` (for non-reaction equations) functions, as well as `unknows` (for all variables) and `equations` (for all equations) functions:

In [ ]:
nutrient_model = @reaction_network begin
    @parameters d
    @equations D(N) ~ - d*Xₐ*N
    (kₐ*N,kᵢ), Xᵢ <--> Xₐ
end

In [ ]:
nonspecies(nutrient_model)

In [ ]:
nonreactions(nutrient_model)

In [ ]:
unknowns(nutrient_model)

In [ ]:
equations(nutrient_model)

#### Basic Programmatic Modelling

When using programmatic modelling, we first declare the model components. Next, we assembly these to a complete model. In the first step, we must fetch the independent variable of the system:

In [ ]:
t = default_t()

Almost always, this is just the time variable (on which other variables depend). However, non-time variables (e.g. when creating spatial models) is also possible. next, we declare the species and parameters. The rules for doing this is basically identical as when the `@species` and `@parameters` options are used in the DSL.

In [ ]:
@parameters k
@species X0(t) X1(t) X2(t) X3(t) # Note that the species are functions of the time independent variable.

Now we create our reactions using the `Reaction` function. It takes the following arguments:
1. The reaction rate.
2. A vector with the substrate species.
3. A vector with the product species.

It can also take two additional arguments.
4. A vector designating the substrate species stoichiometries (by default, `1` is used).
5. A vector designating the product species stoichiometries (by default, `1` is used).

Below we declare the reactions of ou linear pathway model within a single vector.

In [ ]:
rxs = [
    Reaction(k, [X0], [X1]),
    Reaction(k, [X1], [X2]),
    Reaction(k, [X2], [X3]),
]

We can now combine our reactions (and the time independent variable) in a `ReactionSystem` model. We have to pre-append the declaration with `@mtkcomplete` (the reasons for which involve *compositional modelling*, which won't be discussed in this tutorial).

In [ ]:
@mtkcomplete linear_pathway_prog = ReactionSystem(rxs, t)

The programmatic `ReactionSystem` model is for all purposes identical, and we can use it in exactly the same way as the model declared previously through `@reaction-network`:

In [ ]:
u0 = [:X0 => 1.0, :X1 => 0.0, :X2 => 0.0, :X3 => 0.0]
ps = [:k => 0.5]
oprob_dsl = ODEProblem(linear_pathway, u0, 10.0, ps)
osol_dsl = solve(oprob_dsl)
plot(osol_dsl)

In [ ]:
u0 = [:X0 => 1.0, :X1 => 0.0, :X2 => 0.0, :X3 => 0.0]
ps = [:k => 0.5]
oprob_prog = ODEProblem(linear_pathway_prog, u0, 10.0, ps)
osol_prog = solve(oprob_prog)
plot(osol_prog)

#### Generative programmatic modelling

Programmatic modelling is useful in that it permits us to generate arbitrary models without writing down each reactions. let us assume that our linear pathway model was a lot longer:

In [ ]:
linear_pathway_long = @reaction_network begin
    k, X0 --> X1
    k, X1 --> X2
    k, X2 --> X3
    k, X3 --> X4
    k, X4 --> X5
    k, X5 --> X6
    k, X6 --> X7
    k, X7 --> X8
    k, X8 --> X9
    k, X9 --> X10
end

Now, Catalyst supports *vector variables* (and *vector parameters*). Here, we declare a 11-valued vector variable `X`:

In [ ]:
n = 11
@species X(t)[1:n]

Now, we can create all or Reactions using a for-loop:

In [ ]:
rxs = []
for i = 1:10
    push!(rxs, Reaction(k, [X[i]], [X[i+1]]))
end

Next, we create a `ReactionSystem` from this vector of reactions:

In [ ]:
@mtkcomplete lp_long = ReactionSystem(rxs, t)

Finally, when we simulate the model, we give $X$ a vector of values:

In [ ]:
u0_X = zeros(n)
u0_X[1] = 1.0
u0 = [:X => u0_X]
ps = [:k => 0.5]
oprob_prog = ODEProblem(lp_long, u0, 10.0, ps)
osol_prog = solve(oprob_prog)
plot(osol_prog)

In this way, using vector-valued species and parameters, we can easily generate models of arbitrary sizes but with some regular and known structure.

#### Using symbolic variables

When we use e.g. `@species X0(t) X1(t) X2(t) X3(t)` we actually declare `X0`, `X1`, `X2`, and `X3` as *symbolic variables* in our current Julia session. We can check that these are actual variables we can use:

In [ ]:
X1

For a starter, we can now use these as the keys in our parameter maps (i.e. we do not need to use the symbol forms, like `:X0`):

In [ ]:
u0 = [X0 => 1.0, X1 => 0.0, X2 => 0.0, X3 => 0.0]
ps = [k => 0.5]
oprob_prog = ODEProblem(linear_pathway_prog, u0, 10.0, ps)
osol_prog = solve(oprob_prog)
plot(osol_prog)

Similarly, we can use these to designate e.g. which variables to plot:

In [ ]:
plot(osol_prog; idxs = X3)

However, these are *symbolic variables*, which means that these can be used to generate *symbolic expressions*. I.e. we can write their sum:

In [ ]:
X0 + X1 + X2 + X3

Next, we can use this to query our models. I.e. if we do

In [ ]:
plot(osol_prog; idxs = [X0 + X1 + X2 + X3, X3])

We can plot the sum (which we correctly see stays constant throughout the simulation).

In practise, Catalyst models are build upon the Symbolics.jl symbolic modelling package. Internally, Catalyst models are represented entierly symbolically. This means that Catalyst can perform symbolic manipulations to e.g. perform various forms of manipulation. We can expose this in a quick example using Symbolics's symbolic simplification function:

In [ ]:
ex = (X1^2 - X2^2)/(X1 + X2)
Symbolics.simplify(ex)

When modelling using the DSL, we do not typically have access to symbolic variables, and hence use the symbol notation. However, it is possible to indexing into our model to access them:

In [ ]:
rn = @reaction_network begin
    (k1,k2), X1 <--> X2
end
u0 = [rn.X1 => 2.0, rn.X2 => 0.0]
ps = [rn.k1 => 2.0, rn.k2 => 1.0]
oprob = ODEProblem(rn, u0, 2.0, ps)
osol = solve(oprob)
plot(osol)

We can also form symbolic expressions using this syntax:

In [ ]:
plot(osol; idxs = [rn.X1, rn.X1 + rn.X2])

We can also use `@unpack` to unpack these symbolic variables from the model:

In [ ]:
rn = @reaction_network begin
    (k1,k2), X1 <--> X2
end
@unpack X1, X2, k1, k2 = rn
u0 = [X1 => 2.0, X2 => 0.0]
ps = [k1 => 2.0, k2 => 1.0]
oprob = ODEProblem(rn, u0, 2.0, ps)
osol = solve(oprob)
plot(osol; idxs = [X1, X1 + X2])

## Part 2.4 - Modelling using ModelingToolkit

Catalyst is primarily used to make reaction network-based models, however, as we have seen previously, can also integrate equations into its models. Generally, Catalyst is built on the more general ModelingToolkit framework, which is primarily used to create normal equation-based models. if you do not intend to integrate reactions into your models, it is possible to create models directly using ModelingToolkit. Here, ModelingToolkit uses a programmatic syntax that is very similar to Catalyst's.

Below we create a simple Lotka-Volterra predator-prey model. First we import the default time independent variable (`t`) and its differential operator (`D`). Then, we declare the variables (`x` and `y`) and parameters (`α`, `β`, `γ`, `δ`). Next, we can declare our equations. Here, the syntax is the same as we used when declaring non-reaction equations within Catalyst.

In [ ]:
# Import ModelingToolkit.
using ModelingToolkitBase # We use the base version of ModelingToolkit.

# Import time and its differential.
t = default_t() # When using pure ModelingToolkit code, `using ModelingToolkit: t_nounits as t, D_nounits as D` can be used instead.
D = default_time_deriv()

# Declare model components.
@variables x(t) y(t) # ModelingToolkit variables are not species (i.e. cannot partake in reactions). Hence we use `@variables` instead.
@parameters α β γ δ
eqs = [
    D(x) ~ α*x - β*x*y
    D(y) ~ δ*x*y - γ*y
]

While Catalyst calls programmatic models `ReactionSystem`, ModelingToolkit just uses `System`:

In [ ]:
@mtkcomplete lv_model = System(eqs, t)

We can now simulate the model using familiar syntax:

In [ ]:
u0 = [x => 2.0, y => 0.1]
ps = [α => 1.0,  β => 1.0,  γ => 1.0,  δ => 1.0]
oprob = ODEProblem(lv_model, [u0; ps], 20.0) # ModelingToolkit combines the u0 and parameter input as a single array.
osol = solve(oprob)
plot(osol)

Generally, Catalyst's `ReactionSystem` can take normal equations as well, so we can include this input directly in a `ReactionSystem` (which also accepts reactions):

In [ ]:
@mtkcomplete lv_model = ReactionSystem(eqs, t)
u0 = [x => 2.0, y => 0.1]
ps = [α => 1.0,  β => 1.0,  γ => 1.0,  δ => 1.0]
oprob = ODEProblem(lv_model,  u0, 20.0, ps) 
osol = solve(oprob)
plot(osol)